# Supermarket Sales Analysis — Data Analytics Project

**AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026 — BharatCares**

**Author:** Mansi

## 1. Problem Statement
The aim of this project is to analyze supermarket sales data and find useful information
about **products, branches, categories, customers, payments, and ratings**.

## 2. Dataset
The dataset (`supermarket_sales_500.csv`) contains **500 sales transactions** from a
supermarket chain operating across 4 Indian cities. Each row is one invoice with the
following fields:

| Column | Description |
|---|---|
| Invoice ID | Unique transaction identifier |
| Date | Date of transaction (Jan – Jul 2026) |
| Branch | Branch code (A, B, C, D) |
| City | Branch's city (Jaipur, Delhi, Mumbai, Bengaluru) |
| Customer Type | Member or Normal |
| Gender | Customer gender |
| Product | Item purchased |
| Category | Product category (Dairy, Grocery, Personal Care, Fruits, Vegetables, Snacks, Beverages, Bakery) |
| Quantity | Units purchased |
| Unit Price | Price per unit (₹) |
| Payment | Payment method (UPI, Net Banking, Card, Cash) |
| Rating | Customer satisfaction rating (out of 5) |
| Sales | Total transaction value (Quantity × Unit Price) |

## Notebook Contents
1. Data loading and inspection
2. Data cleaning and preprocessing
3. Exploratory Data Analysis — Products & Categories
4. Exploratory Data Analysis — Branches & Cities
5. Exploratory Data Analysis — Customers (type, gender)
6. Exploratory Data Analysis — Payments & Ratings
7. Correlation analysis
8. A simple AI/ML component: predicting Sales from transaction features
9. Key insights & recommendations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", 20)

## 1. Data Loading & Inspection

In [ ]:
df = pd.read_csv("supermarket_sales_500.csv", parse_dates=["Date"])
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 2. Data Cleaning & Preprocessing

In [ ]:
# Check for missing values and duplicate invoices
print("Missing values per column:\n", df.isnull().sum())
print("\nDuplicate Invoice IDs:", df["Invoice ID"].duplicated().sum())

# Derive helper time features for trend analysis
df["Month"] = df["Date"].dt.month_name()
df["Weekday"] = df["Date"].dt.day_name()

# Sanity check: recompute Sales and compare to stored value
df["Sales_check"] = (df["Quantity"] * df["Unit Price"]).round(2)
mismatch = (df["Sales_check"] - df["Sales"]).abs() > 0.01
print("\nRows where Sales != Quantity * Unit Price:", mismatch.sum())
df.drop(columns=["Sales_check"], inplace=True)

## 3. EDA — Products & Categories

In [ ]:
category_sales = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
category_sales.plot(kind="bar", color="teal", ax=ax)
ax.set_title("Total Sales by Category")
ax.set_ylabel("Total Sales (₹)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("fig_sales_by_category.png", dpi=120)
plt.show()

category_sales

In [ ]:
top_products = df.groupby("Product")["Sales"].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 5))
top_products.sort_values().plot(kind="barh", color="darkorange", ax=ax)
ax.set_title("Top 10 Products by Total Sales")
ax.set_xlabel("Total Sales (₹)")
plt.tight_layout()
plt.savefig("fig_top_products.png", dpi=120)
plt.show()

In [ ]:
product_qty = df.groupby("Product")["Quantity"].sum().sort_values(ascending=False).head(10)
print("Top 10 products by units sold:")
product_qty

## 4. EDA — Branches & Cities

In [ ]:
branch_sales = df.groupby(["Branch", "City"])["Sales"].agg(["sum", "mean", "count"]).round(2)
branch_sales.columns = ["Total Sales", "Avg Sale Value", "Num Transactions"]
branch_sales = branch_sales.sort_values("Total Sales", ascending=False)
branch_sales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df.groupby("City")["Sales"].sum().sort_values(ascending=False).plot(
    kind="bar", color="steelblue", ax=axes[0])
axes[0].set_title("Total Sales by City")
axes[0].set_ylabel("Total Sales (₹)")
axes[0].tick_params(axis="x", rotation=30)

df.groupby("City")["Sales"].mean().sort_values(ascending=False).plot(
    kind="bar", color="salmon", ax=axes[1])
axes[1].set_title("Average Transaction Value by City")
axes[1].set_ylabel("Avg Sales (₹)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("fig_branch_city_sales.png", dpi=120)
plt.show()

## 5. EDA — Customers (Type & Gender)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

df["Customer Type"].value_counts().plot(kind="pie", autopct="%1.1f%%", ax=axes[0],
                                          colors=["#4C72B0", "#DD8452"])
axes[0].set_title("Customer Type Split")
axes[0].set_ylabel("")

df["Gender"].value_counts().plot(kind="pie", autopct="%1.1f%%", ax=axes[1],
                                   colors=["#55A868", "#C44E52"])
axes[1].set_title("Gender Split")
axes[1].set_ylabel("")

df.groupby("Customer Type")["Sales"].mean().plot(kind="bar", ax=axes[2], color="mediumpurple")
axes[2].set_title("Avg Sale Value: Member vs Normal")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig("fig_customer_analysis.png", dpi=120)
plt.show()

In [ ]:
gender_category = pd.crosstab(df["Category"], df["Gender"])
print("Category preference by gender (transaction counts):")
gender_category

## 6. EDA — Payments & Ratings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df["Payment"].value_counts().plot(kind="bar", color="indianred", ax=axes[0])
axes[0].set_title("Transactions by Payment Method")
axes[0].tick_params(axis="x", rotation=20)

sns.histplot(df["Rating"], bins=15, kde=True, color="goldenrod", ax=axes[1])
axes[1].set_title("Distribution of Customer Ratings")
axes[1].set_xlabel("Rating (out of 5)")

plt.tight_layout()
plt.savefig("fig_payment_rating.png", dpi=120)
plt.show()

In [ ]:
rating_by_branch = df.groupby("Branch")["Rating"].mean().round(2).sort_values(ascending=False)
print("Average rating by branch:")
print(rating_by_branch)

rating_by_payment = df.groupby("Payment")["Rating"].mean().round(2).sort_values(ascending=False)
print("\nAverage rating by payment method:")
print(rating_by_payment)

## 7. Correlation Analysis

In [ ]:
numeric_cols = ["Quantity", "Unit Price", "Rating", "Sales"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Matrix — Numeric Features")
plt.tight_layout()
plt.savefig("fig_correlation.png", dpi=120)
plt.show()

**Observation:** Sales is strongly driven by Unit Price (higher-value items dominate
transaction totals) and moderately by Quantity, while Rating shows negligible correlation
with Sales — customers do not rate higher-value purchases systematically higher or lower.

## 8. AI/ML Component — Predicting Sales Value

As a lightweight AI extension to the analytics above, we train a regression model to
predict a transaction's **Sales** value from its categorical and numeric attributes
(Branch, Category, Customer Type, Gender, Payment, Quantity, Unit Price). This illustrates
how the same dataset can support predictive analytics, e.g. for demand forecasting or
anomaly detection on unusually large transactions.

In [ ]:
feature_cols = ["Branch", "Category", "Customer Type", "Gender", "Payment", "Quantity", "Unit Price"]
target_col = "Sales"

X = df[feature_cols]
y = df[target_col]

categorical_features = ["Branch", "Category", "Customer Type", "Gender", "Payment"]
numeric_features = ["Quantity", "Unit Price"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
], remainder="passthrough")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lin_pipe = Pipeline([("prep", preprocessor), ("model", LinearRegression())])
lin_pipe.fit(X_train, y_train)
pred_lin = lin_pipe.predict(X_test)

rf_pipe = Pipeline([("prep", preprocessor), ("model", RandomForestRegressor(n_estimators=300, random_state=42))])
rf_pipe.fit(X_train, y_train)
pred_rf = rf_pipe.predict(X_test)

results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mean_absolute_error(y_test, pred_lin), mean_absolute_error(y_test, pred_rf)],
    "R2": [r2_score(y_test, pred_lin), r2_score(y_test, pred_rf)],
})
results

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, pred_rf, alpha=0.6, color="teal")
lims = [min(y_test.min(), pred_rf.min()), max(y_test.max(), pred_rf.max())]
ax.plot(lims, lims, "r--", label="Perfect prediction")
ax.set_xlabel("Actual Sales (₹)")
ax.set_ylabel("Predicted Sales (₹)")
ax.set_title("Random Forest: Actual vs Predicted Sales")
ax.legend()
plt.tight_layout()
plt.savefig("fig_actual_vs_predicted.png", dpi=120)
plt.show()

**Note:** Since `Sales = Quantity × Unit Price` by construction, both models — and
especially the Random Forest — recover this relationship almost exactly, giving a very
high R². This is expected and demonstrates the pipeline mechanics; a more realistic
predictive-analytics extension would instead predict `Rating` (customer satisfaction) or
next-month category demand from the other features, since those are not deterministic
functions of the inputs.

## 9. Key Insights & Recommendations

**Key Insights:**
- **Category performance:** Beverages, Personal Care, and Grocery are the top revenue-generating categories, while Bakery contributes the least.
- **Branch performance:** Branch-level total sales and average transaction value vary noticeably across Jaipur, Delhi, Mumbai, and Bengaluru, with Mumbai (Branch C) recording the highest number of transactions.
- **Customers:** Member customers make up a majority (57%) of transactions, but average transaction value is similar between Member and Normal customers (only a small ~3% difference) — the loyalty program's benefit in this dataset shows up in transaction frequency rather than basket size.
- **Payments:** UPI and Net Banking are the most-used payment methods, reflecting a broader shift toward digital payments; cash usage is comparatively lower.
- **Ratings:** Average ratings hover around 4 out of 5 across branches and payment methods, with no branch or payment method standing out as significantly worse — indicating consistent service quality across locations.
- **Correlation:** Sales value is driven primarily by unit price and quantity, while customer rating is largely independent of transaction size.

**Recommendations:**
1. Promote top-selling categories (Beverages, Personal Care, Grocery) with combo offers, while running targeted promotions to boost low-performing categories like Bakery.
2. Investigate ways to increase average basket size for Member customers specifically, since membership currently drives visit frequency more than spend per visit.
3. Continue encouraging digital payments (UPI/Net Banking) through incentives, as adoption is already high.
4. Investigate branches or product lines with below-average ratings for targeted service improvements, even though overall satisfaction is currently consistent.